# Tập 4.7 · Tương lai: Physical AI (lớp an toàn và mô hình thế giới) 🌐

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap47/tap47_physai.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 4 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Prospects](https://porabook.com/book/prospects/)

Hai ý của chương kết, làm thật nhỏ trên đường đua tập 4.6. Trong notebook này bạn:

1. Xem lớp an toàn cứu mạng BC tệ nhất, và robot lái bằng mô hình thế giới.
2. **Tự viết** khoảng hở hai bên, lớp an toàn và phần tưởng tượng 8 bước (3 bài), có ô chấm tự động (~45 giây).
3. Vẽ số liệu video.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap47")
sys.path[:0] = ["bai_tap", "sim", ".."]

import json
import numpy as np
import an_toan as A               # lớp an toàn của video
import the_gioi as G              # mô hình thế giới của video
from the_gioi import R            # đường đua tập 4.6
from kiem_tra import nap_tg
from ve_do_thi import ve          # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
pi = A.nap("dd_bc2_3.pt")         # mạng BC 2 vòng tệ nhất của tập 4.6
print("BC 2 vòng, seed 3:", R.danh_gia(R.lai_pi(pi), n=100)["xong"], "→ có lớp an toàn:", R.danh_gia(A.boc_an_toan(R.lai_pi(pi)), n=100)["xong"])
print("lái bằng mô hình thế giới (200 đường lái ngẫu nhiên):", R.danh_gia(G.lai_mo(nap_tg()), n=100)["xong"])

## 2. Bài tập

Khoảng hở: $c_L = \min_{\text{tia trái}} d_i \sin a_i$. Lớp an toàn: $c_L < m \Rightarrow \omega \le -k(m - c_L)$. Tưởng tượng: chi phí $= \sum_h 0.9^h\,\text{diem}(\hat o_{h})$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `khoang_ho(obs)` | khoảng hở trái, phải |
| **2** | `boc_an_toan(lai, m, k)` | bọc người lái bằng luật an toàn |
| **3** | `chi_phi_tuong_tuong(f, obs)` | mơ 8 bước cho 25 lệnh, chấm điểm |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np
import torch

from the_gioi import H, W_THU, diem, R  # noqa: F401  (mô hình thế giới của video, trong ../sim; R là đường đua tập 4.6)

GOC_TIA, TAM = R.GOC_TIA, R.TAM


def khoang_ho(obs):
    """Bài 1 (khoảng hở hai bên): mép đường cách robot bao xa theo phương ngang?

    d = obs · TAM (mét). Tia i ở góc a_i chạm mép ở khoảng d_i → khoảng cách ngang tới mép là d_i·|sin a_i|.
    c_L = nhỏ nhất của d_i·sin(a_i) trên các tia TRÁI (a_i > 0.1),
    c_R = nhỏ nhất của d_i·(−sin a_i) trên các tia PHẢI (a_i < −0.1).
    Trả về (c_L, c_R), mỗi cái mảng (B,)."""
    raise NotImplementedError("TODO Bài 1: khoảng hở hai bên")


def boc_an_toan(lai, m=0.17, k=12.0):
    """Bài 2 (lớp an toàn): bọc người lái `lai` (ví dụ mạng nơ-ron) bằng một luật cổ điển. Trả về hàm f(env, obs).

    Trong f: ω = lai(env, obs) (đổi sang mảng float);  c_L, c_R = khoang_ho(obs)
      chỗ nào c_L < m (sát mép trái):  ω ← min(ω, −k·(m − c_L))   (bắt buộc rẽ phải ít nhất chừng đó)
      chỗ nào c_R < m (sát mép phải):  ω ← max(ω, +k·(m − c_R))   (bắt buộc rẽ trái)
    Còn lại giữ nguyên lệnh của mạng. (Gợi ý: np.where, np.minimum, np.maximum.)"""
    raise NotImplementedError("TODO Bài 2: lớp an toàn")


def chi_phi_tuong_tuong(f, obs):
    """Bài 3 (lái bằng tưởng tượng): với mỗi robot, thử 25 lệnh W_THU, "mơ" H = 8 bước bằng mô hình thế giới f, chấm điểm.

    B = số robot. Làm trong torch.no_grad():
      x = obs lặp cho 25 lệnh: tensor (B·25, 9), hàng b·25 + j là obs[b]      (unsqueeze(1).expand(B, 25, 9).reshape(-1, 9))
      w = W_THU lặp B lần: tensor float32 (B·25,), hàng b·25 + j là W_THU[j]  (torch.as_tensor(W_THU, dtype=torch.float32).repeat(B))
      chi_phi = 0;  lặp h = 0..H−1:  x = f(x, w).clamp(0, 1);  chi_phi += diem(x) · 0.9**h
    Trả về chi_phi dạng mảng numpy (B, 25). Lệnh được chọn là W_THU[chi_phi.argmin(1)]."""
    raise NotImplementedError("TODO Bài 3: tưởng tượng 8 bước")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Vùng an toàn rộng hơn hay hẹp hơn thì sao? Thử với mạng DAgger (tốt) và BC 2 vòng (tệ).

In [ ]:
for m in (0.10, 0.17, 0.25):
    kq = {ten: R.danh_gia(A.boc_an_toan(R.lai_pi(A.nap(f)), m=m), n=100)["xong"] for ten, f in [("BC2", "dd_bc2_3.pt"), ("DAgger", "dd_dagger_0.pt")]}
    print(f"m = {m:.2f} m:", kq)

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap47).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)